In [1]:
import pickle
import sys
sys.path.insert(1, '../../../scripts/')
from core.reaction import Metabolic_Reaction, Expression_Reaction

lp_path = '/data2/hratch/human_me/other/test_lp/'

def flatten_list(t):
    #https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-list-of-lists
    return [item for sublist in t for item in sublist]

No objective coefficients in model. Unclear what should be optimized


In [3]:
with open(lp_path + 'working_version_' + str(4) + '.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [53]:
from core.reaction import Metabolic_Reaction, Expression_Reaction
from macromolecules.complex import Complex

class Expressed_Gene():
    '''Tracks all reactions and macromolecules associated with a ME Model gene'''
    def __init__(self, hgnc_id):
        '''
        Init method
        
        Parameters
        ----------
        hgnc_id: str
            the gene HGNC ID
        
        '''
        if not hgnc_id.startswith('HGNC:'):
            raise ValueuError('Currently all genes must be in standard HGNC ID format')
        self.hgnc_id = hgnc_id
        self.reactions = {'Metabolic_Reactions': dict(), 
                          'Expression_Reactions': {'mrna': {'synthesis': None, 'sink': None, 'other': []}, 
                                                  'protein': {'translation': None, 'synthesis': None, 'sink': None, 'other': []}, 
                                                  'complex': {'synthesis': None, 'sink': None, 'other': []}}}
        
    def add_metabolic_reaction(self, r):
        '''
        
        Adds metabolic reaction associated with gene. 
        
        Hierarchy is organized as follows: {reaction_id: {catalysis: enzyme_id, deg_proxy: proxy_id}}
        
        Where catalysis: enzyme_id represents the macromolecules enzyme that is coupled to the reaction for 
        protein synthesis to reaction catalysis and deg_proxy: proxy_id represents the proxy macromolecule that couples 
        protein degradation to reaction catalysis. 
        
        '''
        if not isinstance(r, Metabolic_Reaction):
            raise ValueError(r.id + ' is not a metabolic reaction')
        
        assoc_macro = dict()
        assoc_macro = {t: m for m,t in r.coupled_metabolites.items()}
        is_complex = False
        if isinstance(assoc_macro['catalysis'], Complex):
            is_complex = True
        for m,t in r.coupled_metabolites.items():
            if t not in ['catalysis', 'enzyme_degradation']:
                raise ValueError('Unexpected coupling type in metabolic reaction for ' + r.id)
            if not is_complex and m.hgnc_id != self.hgnc_id:
                raise ValueError('The macromolecule does not have the appropriate hgnc id: ' + m.id)
            elif is_complex:
                raise ValueError('Complexes not yet dealt with')
            assoc_macro[m.id] = t
        self.reactions['Metabolic_Reactions'][r.id] = assoc_macro
    
    def add_expression_reaction(self, r):
        if not isinstance(r, Expression_Reaction):
            raise ValueError(r.id + ' is not an expression reaction')
        if hasattr(r, 'hgnc_id') and r.hgnc_id != self.hgnc_id:
            raise ValueError(r.id + " 's .hgnc_id attribute does not match the Expressed_Gene hgnc_id attribute'")
        
        
        if r.subsystem == 'mRNA_expression':
            key = 'mrna'
        elif r.subsystem.startswith('Protein_'):
            key = 'protein'
        elif r.subsystem.startswith('Complex_'):
            key = 'complex'
        else:
            raise ValueError('Unaccounted for Expression Reaction subsystem')
            
            
        if r.synthesis:
            if self.reactions['Expression_Reactions'][key]['synthesis'] is not None:
                raise ValueError('Multiple mRNA synthesis reactions assigned to ' + self.hgnc_id)
            self.reactions['Expression_Reactions'][key]['synthesis'] = r.id
        elif r.sink:
            if self.reactions['Expression_Reactions'][key]['sink'] is not None:
                raise ValueError('Multiple mRNA sink reactions assigned to ' + self.hgnc_id)
            self.reactions['Expression_Reactions'][key]['sink'] = r.id
        else:
            self.reactions['Expression_Reactions'][key]['other'] = r.id
    def check(self):
        # all proteins have a translation reaction
        # appropriate couplings between reactions
            

In [5]:
metabolic_reactions = [r for r in me_model.reactions if isinstance(r, Metabolic_Reaction)]

In [55]:
mr = [r for r in me_model.reactions if isinstance(r, Metabolic_Reaction)][0]
hgnc_id = 'HGNC:23408'
g = Expressed_Gene(hgnc_id = hgnc_id)
g.add_metabolic_reaction(mr)

ers = [r for r in me_model.reactions if isinstance(r, Expression_Reaction) and r.hgnc_id == hgnc_id]

for r in ers:
    g.add_expression_reaction(r)

In [56]:
g.reactions

{'Metabolic_Reactions': {'3HBCDm_F': {'catalysis': <Protein HGNC:23408_folded_pre_protein_m at 0x7f1ba3af8e48>,
   'enzyme_degradation': <Macromolecule HGNC:23408_enzyme_deg_proxy at 0x7f1ba3a93358>,
   'HGNC:23408_folded_pre_protein_m': 'catalysis',
   'HGNC:23408_enzyme_deg_proxy': 'enzyme_degradation'}},
 'Expression_Reactions': {'mrna': {'synthesis': 'HGNC:23408_TRANSCRIPTION',
   'sink': 'HGNC:23408_DECAPPING_mRNA_DEGRADATIONc',
   'other': 'HGNC:23408_TRANSCRIPTION'},
  'protein': {'synthesis': 'HGNC:23408_TRANSLATION_ELONGATIONc',
   'sink': 'HGNC:23408_DEGRADATIONm',
   'other': 'HGNC:23408_IMPORTtm'},
  'complex': {'synthesis': None, 'sink': None, 'other': []}}}

In [58]:
set([r.subsystem for r in me_model.reactions if isinstance(r, Expression_Reaction)])

{'Complex_Degradation',
 'Complex_Formation',
 'Protein_Degradation',
 'Protein_Expression',
 'mRNA_expression',
 'rRNA_expression',
 'tRNA_Biogenesis'}

['Protein_Expression', 'Protein_Expression', 'Protein_Degradation']

In [16]:
mrs = set(flatten_list([list(r.coupled_metabolites.values()) for r in me_model.reactions if isinstance(r, Metabolic_Reaction)]))


In [17]:
mrs

{'catalysis', 'enzyme_degradation'}